In [1]:
import pandas as pd
import osmnx as ox
import folium
import h3

df = pd.read_parquet('geo_2025-02-03.parquet')

df['LocalTime'] = pd.to_datetime(df['LocalTime'])
df = df.sort_values(by=['DeviceID', 'LocalTime'])

target_users = df['DeviceID'].unique()[:3]

print('выбранные пользователи', target_users)

for user_id in target_users:
    print(f'анализ для {user_id}')
    
    user_df = df[df['DeviceID'] == user_id].copy()
    
    mean_lat = user_df['Latitude'].mean()
    mean_lng = user_df['Longitude'].mean()
    
    tags = {'amenity': True, 'shop': True, 'leisure': True}
    pois = ox.features_from_point((mean_lat, mean_lng), tags, 500)
    
    poi_types = []
    for tag in tags:
        if tag in pois.columns:
            poi_types.extend(pois[tag].dropna().tolist())
            
    if poi_types:
        top_pois = pd.Series(poi_types).value_counts().head(3)
        print('частые типы объектов рядом:')
        for p_type, count in top_pois.items():
            print(f'- {p_type}: {count} раз')
    else:
        print('объекты рядом не найдены')

    graph = ox.graph_from_point((mean_lat, mean_lng), 1000, network_type='drive')
    
    sample_points = user_df.iloc[::10]
    nearest_edges = ox.nearest_edges(graph, sample_points['Longitude'], sample_points['Latitude'])
    
    top_edges = pd.Series(nearest_edges).value_counts().head(3)
    
    print('наиболее используемые дороги:')

    for (u, v, k), count in top_edges.items():
        edge_data = graph.get_edge_data(u, v, k)
        print(f"- {edge_data.get('name', 'без названия')}")

    start_lat = user_df['Latitude'].iloc[0]
    start_lng = user_df['Longitude'].iloc[0]
    m = folium.Map([start_lat, start_lng], zoom_start=13)
    
    points = user_df[['Latitude', 'Longitude']].values.tolist()
    folium.PolyLine(points).add_to(m)
    
    user_df['h3_index'] = user_df.apply(lambda r: h3.latlng_to_cell(r['Latitude'], r['Longitude'], 9), axis=1)
    hex_indices = user_df['h3_index'].unique()
    
    for h3_idx in hex_indices:
        boundaries = h3.cell_to_boundary(h3_idx)
        folium.Polygon(boundaries).add_to(m)
        
    m.save(f'user_{user_id}_route_map.html')
    print(f'карта созранена в user_{user_id}_route_map.html')

выбранные пользователи [107225 257464 263597]
анализ для 107225


частые типы объектов рядом:
- parking: 9 раз
- outpost: 6 раз
- kindergarten: 5 раз


наиболее используемые дороги:
- улица Марковцева
карта созранена в user_107225_route_map.html
анализ для 257464


частые типы объектов рядом:
- school: 1 раз
- townhall: 1 раз
- hospital: 1 раз


наиболее используемые дороги:
- Комсомольская улица
- Комсомольская улица
карта созранена в user_257464_route_map.html
анализ для 263597


частые типы объектов рядом:
- parking: 45 раз
- playground: 22 раз
- pitch: 16 раз


наиболее используемые дороги:
- улица Светланова
- улица Шувалова
- улица Светланова
карта созранена в user_263597_route_map.html
